In [1]:
%pip install gluonts


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.0/864.0 kB 20.8 MB/s eta 0:00:00


In [2]:
!git clone -b update-gluonts https://github.com/time-series-foundation-models/lag-llama/


Cloning into 'lag-llama'...
remote: Enumerating objects: 508, done.
remote: Counting objects: 100% (183/183), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 508 (delta 155), reused 114 (delta 114), pack-reused 325 (from 3)
Receiving objects: 100% (508/508), 286.89 KiB | 5.85 MiB/s, done.
Resolving deltas: 100% (253/253), done.


In [3]:
%cd /content/lag-llama


/content/lag-llama


In [4]:
!pip install -r requirements.txt


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.0/45.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 849.3/849.3 kB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.0/853.0 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.2 MB/s eta 0:00:00


In [5]:
!pip install -U torch torchvision


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 60.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 6.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-cusparselt-cu13
    Found existing installation: nvidia-cusparselt-cu13 0.8.0
    Uninstalling nvidia-cusparselt-cu13-0.8.0:
      Successfully uninstalled nvidia-cusparselt-cu13-0.8.0
  Attempting uninstall: cuda-toolkit
    Found existing installation: cuda-toolkit 13.0.2
    Uninstalling cuda-toolkit-13.0.2:
      Successfully uninstalled cuda-toolkit-13.0.2
  Attempting uninstall: triton
    Found existin

**Restart**

In [1]:
%cd /content/lag-llama


/content/lag-llama


In [2]:
!hf download time-series-foundation-models/Lag-Llama lag-llama.ckpt --local-dir /content/lag-llama


Hint: A new version of huggingface_hub (2.2.0) is available! You are using version 1.33.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g` to teach your AI agents how to use the `hf` CLI.

lag-llama.ckpt: downloading bytes:  89% 26.3M/29.5M [00:01<00:00, 39.1MB/s, 1.70MB/s  ]
lag-llama.ckpt: downloading bytes: 100% 27.8M/27.8M [00:01<00:00, 20.9MB/s, 2.65MB/s  ]
lag-llama.ckpt: reconstructing file: 100% 29.5M/29.5M [00:01<00:00, 22.2MB/s, 2.85MB/s  ]
✓ Downloaded
  path: /content/lag-llama/lag-llama.ckpt


In [3]:
import json
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from gluonts.dataset.pandas import PandasDataset
from gluonts.evaluation import make_evaluation_predictions
from lag_llama.gluon.estimator import LagLlamaEstimator

In [4]:
%pip install sktime
from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sktime.performance_metrics.forecasting import MeanAbsoluteScaledError

drive.mount('/content/drive')

CONFIG_PATH = '/content/drive/MyDrive/Models parameters JSON files/seasonalnaive.json'
OUTPUT_DIR = '/content/drive/MyDrive/results/json/Lag_Llama'
TRAIN_RATIO = 0.80
NUM_SAMPLES = 100
context_len = 32
SHOW_PLOTS = True

with open(CONFIG_PATH, encoding='utf-8') as file:
    CONFIG = json.load(file)

DATASETS = CONFIG['datasets']
total_targets = sum(len(dataset['targets']) for dataset in DATASETS)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Loaded {len(DATASETS)} datasets and {total_targets} targets')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 17.3 MB/s eta 0:00:00
Mounted at /content/drive
Loaded 6 datasets and 26 targets


In [5]:
# @title
import sys
from types import ModuleType

# Create dummy module hierarchy
def create_dummy_module(module_path):
    """
    Create a dummy module hierarchy for the given path.
    Returns the leaf module.
    """
    parts = module_path.split('.')
    current = ''
    parent = None

    for part in parts:
        current = current + '.' + part if current else part
        if current not in sys.modules:
            module = ModuleType(current)
            sys.modules[current] = module
            if parent:
                setattr(sys.modules[parent], part, module)
        parent = current

    return sys.modules[module_path]

# Create the dummy gluonts module hierarchy
gluonts_module = create_dummy_module('gluonts.torch.modules.loss')

# Create dummy classes for the specific loss functions
class DistributionLoss:
    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, *args, **kwargs):
        return 0.0

    def __getattr__(self, name):
        return lambda *args, **kwargs: None

class NegativeLogLikelihood:
    def __init__(self, *args, **kwargs):
        pass

    def __call__(self, *args, **kwargs):
        return 0.0

    def __getattr__(self, name):
        return lambda *args, **kwargs: None

# Add the specific classes to the module
gluonts_module.DistributionLoss = DistributionLoss
gluonts_module.NegativeLogLikelihood = NegativeLogLikelihood


In [6]:
# Allow the checkpoint classes required by Lightning's weights-only loader.
from gluonts.torch.distributions.studentT import StudentTOutput

torch.serialization.add_safe_globals([
    StudentTOutput,
    (DistributionLoss, 'gluonts.torch.modules.loss.DistributionLoss'),
    (NegativeLogLikelihood, 'gluonts.torch.modules.loss.NegativeLogLikelihood'),
])


##scaled metrics

In [7]:
from sklearn.preprocessing import MinMaxScaler


def scale_values(y_train, y_true, y_pred):
    y_train = np.asarray(y_train, dtype=np.float64).reshape(-1, 1)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1, 1)
    y_pred = np.asarray(y_pred, dtype=np.float64).reshape(-1, 1)

    scaler = MinMaxScaler()
    scaler.fit(y_train)
    y_true_scaled = scaler.transform(y_true)
    y_pred_scaled = scaler.transform(y_pred)

    return y_true_scaled, y_pred_scaled


##CRPS metric

In [8]:
def pinball_loss(q, y, alpha):
    return (alpha - (y < q)) * (y - q)


def crps_Lag_llama_quantile_version(samples, y_true):
    samples = np.array(samples)
    y_true = np.array(y_true)

    alphas = np.arange(0.1, 1.0, 0.1)
    q_preds = {
        alpha: np.quantile(samples, alpha, axis=0)
        for alpha in alphas
    }

    denominator = np.sum(np.abs(y_true))
    wql_values = []

    for alpha in alphas:
        q = q_preds[alpha]
        pin = pinball_loss(q, y_true, alpha)
        wql = 2 * np.sum(pin) / denominator
        wql_values.append(wql)

    return float(np.mean(wql_values))


## Save to JSON

In [9]:
def save_timeseries_full(
    log_path, model_name, full_dates, full_values, train_dates, train_values,
    test_dates, test_values, forecast_dates, forecast_quantiles, metrics
):
    if os.path.exists(log_path):
        with open(log_path, 'r', encoding='utf-8') as file:
            log_data = json.load(file)
    else:
        log_data = []

    forecast_data = {'dates': [str(date) for date in forecast_dates]}
    for name, values in forecast_quantiles.items():
        forecast_data[name] = [float(value) for value in values]

    entry = {
        'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'model': model_name,
        'full_series': {
            'dates': [str(date) for date in full_dates],
            'values': [float(value) for value in full_values],
        },
        'train_series': {
            'dates': [str(date) for date in train_dates],
            'values': [float(value) for value in train_values],
        },
        'test_series': {
            'dates': [str(date) for date in test_dates],
            'values': [float(value) for value in test_values],
        },
        'forecast': forecast_data,
        'metrics': {
            name: None if not np.isfinite(value) else float(value)
            for name, value in metrics.items()
        },
    }

    log_data.append(entry)
    with open(log_path, 'w', encoding='utf-8') as file:
        json.dump(log_data, file, indent=4, ensure_ascii=False)


## Model-specific forecast function

In [10]:
# @title
def get_lag_llama_predictions(dataset, prediction_length, device, context_length=context_len, use_rope_scaling=False, num_samples=100):
    # Change map_location to 'cpu' to load the model onto the CPU.
    ckpt = torch.load("lag-llama.ckpt", map_location=torch.device('cuda'), weights_only=False) # Uses GPU since in this Colab we use a GPU.
    estimator_args = ckpt["hyper_parameters"]["model_kwargs"]

    rope_scaling_arguments = {
        "type": "linear",
        "factor": max(1.0, (context_length + prediction_length) / estimator_args["context_length"]),
    }

    estimator = LagLlamaEstimator(
        ckpt_path="lag-llama.ckpt",
        prediction_length=prediction_length,
        context_length=context_length, # Lag-Llama was trained with a context length of 32, but can work with any context length

        # estimator args
        input_size=estimator_args["input_size"],
        n_layer=estimator_args["n_layer"],
        n_embd_per_head=estimator_args["n_embd_per_head"],
        n_head=estimator_args["n_head"],
        scaling=estimator_args["scaling"],
        time_feat=estimator_args["time_feat"],
        rope_scaling=rope_scaling_arguments if use_rope_scaling else None,

        batch_size=1,
        num_parallel_samples=100,
        device=device,
    )

    lightning_module = estimator.create_lightning_module()
    transformation = estimator.create_transformation()
    predictor = estimator.create_predictor(transformation, lightning_module)

    forecast_it, _ = make_evaluation_predictions(
        dataset=dataset,
        predictor=predictor,
        num_samples=num_samples
    )
    forecasts = list(forecast_it)
    return forecasts


## Prepare one target column

In [11]:
def prepare_target(data, dataset, target):
    dataset_name = dataset['dataset_name']
    time_column = dataset['time_column']
    target_column = target['column']

    target_data = data[[time_column, target_column]].copy()
    target_data[target_column] = pd.to_numeric(target_data[target_column])

    dates = target_data[time_column]
    values = target_data[target_column].to_numpy(dtype=float)

    return dataset_name, time_column, target_column, dates, values


## Split data into training and test parts

In [12]:
def split_data(dates, values):
    split_index = int(len(values) * TRAIN_RATIO)

    train_dates = dates.iloc[:split_index]
    train_values = values[:split_index]
    test_dates = dates.iloc[split_index:]
    test_values = values[split_index:]

    return train_dates, train_values, test_dates, test_values


## Create a Lag-Llama forecast

In [13]:
def create_forecast(dates, train_values, test_values, num_samples=NUM_SAMPLES):
    frequency = pd.infer_freq(dates)
    all_values = np.concatenate([train_values, test_values]).astype(np.float32)
    frame = pd.DataFrame(
        {'target': all_values, 'item_id': 'series_1'},
        index=pd.DatetimeIndex(dates),
    )
    dataset = PandasDataset.from_long_dataframe(
        frame, item_id='item_id', target='target', freq=frequency
    )

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    forecasts = get_lag_llama_predictions(
        dataset=dataset,
        prediction_length=len(test_values),
        device=device,
        context_length=len(train_values),
        use_rope_scaling=False,
        num_samples=num_samples,
    )
    samples = np.asarray(forecasts[0].samples, dtype=np.float64)
    quantiles = {
        'prediction': np.quantile(samples, 0.50, axis=0),
        'median': np.quantile(samples, 0.50, axis=0),
        'p5': np.quantile(samples, 0.05, axis=0),
        'p10': np.quantile(samples, 0.10, axis=0),
        'p25': np.quantile(samples, 0.25, axis=0),
        'p75': np.quantile(samples, 0.75, axis=0),
        'p90': np.quantile(samples, 0.90, axis=0),
        'p95': np.quantile(samples, 0.95, axis=0),
    }

    return samples, quantiles


## Calculate forecast metrics

In [14]:
def calculate_metrics(train_values, test_values, predictions, samples):
    test_array = np.asarray(test_values, dtype=np.float64)
    prediction_array = np.asarray(predictions, dtype=np.float64)

    mase = MeanAbsoluteScaledError()(
        test_array, prediction_array, y_train=train_values
    )

    # Unscaled errors use the original values.
    mae = float(mean_absolute_error(test_array, prediction_array))
    mse = float(mean_squared_error(test_array, prediction_array))
    rmse = float(np.sqrt(mse))

    # Scaled errors use a MinMaxScaler fitted only on the training values.
    test_scaled, predictions_scaled = scale_values(
        train_values, test_array, prediction_array
    )
    scaled_mae = float(mean_absolute_error(test_scaled, predictions_scaled))
    scaled_mse = float(mean_squared_error(test_scaled, predictions_scaled))
    scaled_rmse = float(np.sqrt(scaled_mse))

    metrics = {
        'mae': mae,
        'mse': mse,
        'rmse': rmse,
        'mase': float(mase),
        'mae_scaled': scaled_mae,
        'mse_scaled': scaled_mse,
        'rmse_scaled': scaled_rmse,
        'crps': crps_Lag_llama_quantile_version(samples, test_values),
    }

    return metrics


## Draw the forecast chart

In [15]:
def show_forecast(train_dates, train_values, test_dates, test_values, quantiles,
                  dataset_name, time_column, target_column):
    if not SHOW_PLOTS:
        return

    plt.figure(figsize=(16, 6))
    plt.plot(train_dates, train_values, label='Training data')
    plt.plot(test_dates, test_values, label='Actual values')
    plt.plot(test_dates, quantiles['median'], '--', label='Lag-Llama median')
    plt.fill_between(test_dates, quantiles['p25'], quantiles['p75'], alpha=0.25, label='50% interval')
    plt.fill_between(test_dates, quantiles['p10'], quantiles['p90'], alpha=0.15, label='80% interval')
    plt.title(f'{dataset_name} - {target_column}')
    plt.xlabel(time_column)
    plt.ylabel(target_column)
    plt.legend()
    plt.grid()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    plt.close()


## Save one result as JSON

In [16]:
def save_result(dataset_name, target_column, target, dates, values,
                train_dates, train_values, test_dates, test_values,
                quantiles, metrics):
    output_file = target.get(
        'output_file', f'{dataset_name}_{target_column}.json'
    )
    output_file = output_file.replace('/', '_').replace('\\', '_').replace(':', '_')
    log_path = os.path.join(OUTPUT_DIR, output_file)

    save_timeseries_full(
        log_path,
        'Lag-Llama',
        dates,
        values,
        train_dates,
        train_values,
        test_dates,
        test_values,
        test_dates,
        quantiles,
        metrics,
    )

    result = {
        'dataset': dataset_name,
        'target': target_column,
        'model': 'Lag-Llama',
        'output_file': output_file,
    }
    result.update(metrics)
    all_results.append(result)

    print('JSON saved:', log_path)


## Run all steps for one target column

In [17]:
def run_target(data, dataset, target, model_number):
    dataset_name, time_column, target_column, dates, values = prepare_target(
        data, dataset, target
    )

    train_dates, train_values, test_dates, test_values = split_data(dates, values)
    samples, quantiles = create_forecast(dates, train_values, test_values)
    predictions = quantiles['median']
    metrics = calculate_metrics(train_values, test_values, predictions, samples)

    print(f"\nModel {model_number}/{total_models}: {dataset_name} - {target_column}")
    print('Scaled MAE:', round(metrics['mae_scaled'], 4))
    print('Scaled RMSE:', round(metrics['rmse_scaled'], 4))
    print('MASE:', round(metrics['mase'], 4))

    show_forecast(
        train_dates, train_values, test_dates, test_values, quantiles,
        dataset_name, time_column, target_column,
    )

    save_result(
        dataset_name, target_column, target, dates, values,
        train_dates, train_values, test_dates, test_values, quantiles, metrics,
    )


## Run all datasets and target columns

In [18]:
total_models = sum(len(dataset['targets']) for dataset in DATASETS)
all_results = []

for dataset in DATASETS:
    dataset_name = dataset['dataset_name']
    file_path = dataset['file_path']
    time_column = dataset['time_column']

    data = pd.read_csv(file_path)
    data[time_column] = pd.to_datetime(data[time_column])
    data = data.sort_values(time_column).reset_index(drop=True)

    print('\n' + '=' * 60)
    print('Dataset:', dataset_name)

    for target in dataset['targets']:
        run_target(data, dataset, target, len(all_results) + 1)


Output hidden; open in https://colab.research.google.com to view.

In [19]:
print('\n\n')
print('=' * 110)
print('ALL DATASETS / TARGETS COMPLETED')
print('=' * 110)
if len(all_results) > 0:
    summary_df = pd.DataFrame(all_results)
    print(summary_df.to_string(index=False))
else:
    print('No target was completed.')

print('\n')
print('=' * 90)
print('PROCESS FINISHED')
print('=' * 90)
print('Datasets requested:', len(DATASETS))
print('Targets requested:', total_models)
print('Targets completed:', len(all_results))
print('Output directory:', OUTPUT_DIR)
if len(all_results) != total_models:
    print('\nWARNING:')
    print('Not all requested targets completed successfully.')
    print('Requested:', total_models)
    print('Completed:', len(all_results))
else:
    print('ALL REQUESTED TARGETS COMPLETED.')





ALL DATASETS / TARGETS COMPLETED
dataset         target     model               output_file           mae          mse          rmse      mase  mae_scaled  mse_scaled  rmse_scaled     crps
   WB90             wt Lag-Llama                WB_wt.json      0.395914 2.554381e-01      0.505409  2.945854    0.145557    0.034526     0.185812 0.025218
   WB90            toc Lag-Llama               WB_toc.json      0.067969 7.823467e-03      0.088450  0.794223    0.144615    0.035416     0.188192 0.024931
   WB90    temperature Lag-Llama       WB_temperature.json      2.216579 9.216507e+00      3.035870  3.915857    0.233570    0.102337     0.319902 0.074468
   WB90             ph Lag-Llama                WB_ph.json      0.103832 1.513034e-02      0.123005  1.450492    0.324475    0.147757     0.384392 0.017771
   WB90       humidity Lag-Llama          WB_humidity.json     14.154763 2.904038e+02     17.041238  5.987465    0.301165    0.131464     0.362580 0.180045
   WB90             do Lag-L